In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as ss
from tsflex.features import (
    FeatureCollection,
    FeatureDescriptor,
    MultipleFeatureDescriptors,
    FuncWrapper,
)
# from utils.feature_selection import select_features 
import plotly.graph_objects as go
from scipy.interpolate import griddata
from plotly_resampler import FigureResampler, FigureWidgetResampler
from plotly.subplots import make_subplots
import itertools

from typing import List
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler, PowerTransformer
from sklearn.linear_model import SGDRegressor, HuberRegressor, LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor

from functional import seq
import re

c:\Users\Administrator\Desktop\winder-slip\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load files
loc_data_dir = Path("../data/processed")
df_train = pd.read_parquet(loc_data_dir / "processed_train.parquet")
df_validation = pd.read_parquet(loc_data_dir / "processed_validation.parquet")
df_test = pd.read_parquet(loc_data_dir / "processed_test.parquet")
set_dict = {'Train': df_train, 'Validation': df_validation, 'Test': df_test}

In [3]:
TARGET_COL = "slip"
TIME_COL = "time_idx"
GROUP_COL = "File"

## Compute the target variable
RADIUS = 0.074
df_train[TARGET_COL] = (
    -df_train["Encoders.Encoder1.Speed"] * 2 * np.pi * RADIUS / 2
    - df_train["Motors.Traction1.Speed"]
)
df_validation[TARGET_COL] = (
    -df_validation["Encoders.Encoder1.Speed"] * 2 * np.pi * RADIUS / 2
    - df_validation["Motors.Traction1.Speed"]
)
df_test[TARGET_COL] = (
    -df_test["Encoders.Encoder1.Speed"] * 2 * np.pi * RADIUS / 2
    - df_test["Motors.Traction1.Speed"]
)

# Set the time index column its name
df_train.index.name = TIME_COL
df_validation.index.name = TIME_COL
df_test.index.name = TIME_COL

# make the grouping column a category
df_train[GROUP_COL] = df_train[GROUP_COL].astype('category')
df_validation[GROUP_COL] = df_validation[GROUP_COL].astype('category')
df_test[GROUP_COL] = df_test[GROUP_COL].astype('category')

In [4]:
all_df = pd.concat([df_train, df_validation, df_test])

In [5]:
x = []
y = []
z = []
for id, g in all_df.groupby(['PulseSpeed', 'PulseRampTime', 'PulseDirection']):
    x.append(id[0] * id[2]) 
    y.append(id[1])
    z.append(g['Slip'].abs().mean())

In [6]:
X_raw = np.array(x).flatten()
Y_raw = np.array(y).flatten()
Z_raw = np.array(z).flatten()

x_vec = np.linspace(np.min(X_raw), np.max(X_raw), 100)
y_vec = np.linspace(np.min(Y_raw), np.max(Y_raw), 100)
grid_x, grid_y = np.meshgrid(x_vec, y_vec)

Z_grid = griddata((X_raw, Y_raw), Z_raw, (grid_x, grid_y), method='linear')

fig = go.Figure()

fig.add_trace(go.Contour(
    z=Z_grid,
    x=x_vec, # 1D x-axis
    y=y_vec, # 1D y-axis
    colorscale='Viridis',
    contours_coloring='heatmap', 
    line_width=0,                
    colorbar=dict(title="| <span style='text-decoration:overline'>S</span><sub>actual</sub> | <br>(m/min)"),
    hoverinfo='skip'            
))

fig.add_trace(go.Scatter(
    x=X_raw,
    y=Y_raw,
    mode='markers',
    marker=dict(
        color=Z_raw,
        colorscale='Viridis',
        size=10,
        line=dict(color='white', width=0.5),
        showscale=False # Hide second colorbar
    ),
    name="Original Data"
))

fig.add_vline(x=51.2, line=dict(color="rgba(255, 0, 0, 0.8)", width=2.5, dash="dash"))
fig.add_vline(x=-51.2, line=dict(color="rgba(255, 0, 0, 0.8)", width=2.5, dash="dash"))


fig.add_annotation(
    text="All",
    # xref="paper", yref="paper",
    x=0, y=7,
    showarrow=False,
    font=dict(size=20, color="rgba(255, 0, 0, 0.8)"),
)


fig.add_annotation(
    text="Test",
    x=56, y=7,
    showarrow=False,
    font=dict(size=20, color="rgba(255, 0, 0, 0.8)"),
)

fig.add_annotation(
    text="Test",
    x=-56, y=7,
    showarrow=False,
    font=dict(size=20, color="rgba(255, 0, 0, 0.8)"),
)

fig.update_layout(
    xaxis_title="Web speed (m/min)",
    yaxis_title="Ramp (s)",
    template="plotly_white",
    width=1000,
    height=400
)

fig.update_xaxes(tickvals=np.unique(X_raw))
fig.update_yaxes(tickvals=np.unique(Y_raw))

fig.show()
# Save image as pdf 
fig.write_image("./../figures/slip_contour_plot.pdf")

In [7]:
def process_signals(df: pd.DataFrame):
    # based on the formula
    df['Motors.Roller1.WebSpeedSimple'] = df['Motors.Roller1.Speed']*df['AnalogSensors.Radius1Fine']*2*np.pi
    df['Motors.Roller1.SlipSimple'] = df['Motors.Roller1.WebSpeedSimple'] - df['Motors.Traction1.Speed']

    df['Roller1_diff(speed)'] = df['Motors.Roller1.Speed'].diff().fillna(0)
    df['delta_traction_speed'] = df['Motors.Traction1.Speed'] - df['Motors.Traction2.Speed']

    # data from the sensors
    # Traction 1
    df["Motors.Traction1.Setpoint_diff"] = df["Motors.Traction1.Setpoint"].diff().fillna(0)
    df["Motors.Traction1.Position_diff"] = df["Motors.Traction1.Position"].diff().fillna(0)

    df["Traction1_diff(delta(torque, speed))"] = (df["Motors.Traction1.Torque"] - df["Motors.Traction1.Speed"]).diff().fillna(0)
    df["Traction1_delta(torque,diff(speed))"] = df["Motors.Traction1.Torque"] - df["Motors.Traction1.Speed"].diff().fillna(0)
    df["Traction1_diff(speed)"] = df["Motors.Traction1.Speed"].diff().fillna(0)

    # data from the sensors
    # Traction 2
    df["Motors.Traction2.Setpoint_diff"] = df["Motors.Traction2.Setpoint"].diff().fillna(0)
    df["Motors.Traction2.Position_diff"] = df["Motors.Traction2.Position"].diff().fillna(0)
    df["Traction2_diff(delta(torque, speed))"] = (df["Motors.Traction2.Torque"] - df["Motors.Traction2.Speed"]).diff().fillna(0)
    df["Traction2_delta(torque,diff(speed))"] = df["Motors.Traction2.Torque"] - df["Motors.Traction2.Speed"].diff().fillna(0)

    # Dancer
    df["Dancer1_delta(torque,diff(speed))"] = df["Motors.Dancer1.Torque"] - df["Motors.Dancer1.Speed"].diff().fillna(0)
    df["Dancer1_delta(torque, speed)"] = df["Motors.Dancer1.Torque"] - df["Motors.Dancer1.Speed"]
    df["Dancer1_diff(delta(torque, speed))"] = (df["Motors.Dancer1.Torque"] - df["Motors.Dancer1.Speed"]).diff().fillna(0)
    df["Dancer_diff(speed)"] = df["Motors.Dancer1.Speed"].diff().fillna(0)
    df["Dancer_diff(position)"] = df["Motors.Dancer1.Position"].diff().fillna(0)

    # Loadcell
    df["LoadCell1_diff"] = df["AnalogSensors.LoadCell1"].diff().fillna(0)
    df["LoadCell2_diff"] = df["AnalogSensors.LoadCell2"].diff().fillna(0)

    df["delta(LoadCell1, LoadCell2)"] = df["AnalogSensors.LoadCell1"] - df["AnalogSensors.LoadCell2"]
    df["diff(delta(LoadCell1, LoadCell2))"] = df["delta(LoadCell1, LoadCell2)"].diff().fillna(0)

    df["avg_traction1_accel"] = df['Motors.Traction1.Speed'].diff().fillna(0).abs().rolling(10, center=True).mean()
    if TARGET_COL in df.columns:
        df['slip_compensated_acc'] = df['slip'] / (1 + df['avg_traction1_accel'])


    df["avg_traction1_accel"] = (
        df["Motors.Traction1.Speed"]
        .diff()
        .fillna(0)
        .abs()
        .rolling(10, center=True)
        .mean()
    )

    # Gain factor
    # df["gain_factor"] = -((11 - df["PulseRampTime"]) * df["PulseSpeed"])
    # # Add multiple the gain factor to all current signals
    # for col in df.columns:
    #     if col not in ["gain_factor"] + [TIME_COL, GROUP_COL, TARGET_COL] + ['Time', 'Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']:
    #         df[f"{col}_gain"] = df[col] * df["gain_factor"]
    return df

In [8]:
df_train = process_signals(df_train)
df_validation = process_signals(df_validation)
df_test = process_signals(df_test)

In [9]:
def abs_mean_std_vect(x: np.ndarray):
    x_abs = np.abs(x)
    return np.mean(x_abs, axis=-1), np.std(x_abs, axis=-1)

def std_vect(x: np.ndarray):
    return np.std(x, axis=-1)

def mean_vect(x: np.ndarray):
    return np.mean(x, axis=-1)

def min_max_vect(x: np.ndarray):
    return np.min(x, axis=-1), np.max(x, axis=-1)

def quantiles_vect(
    x: np.ndarray,
    qs: List[float],
    add_ptp: bool = False,
    add_iqr: bool = False,
    axis=-1,
) -> np.ndarray:
    """Get the quantiles of a 1D signal (vectorized)

    Parameters
    ----------
    x : np.ndarray
        The input 1D signal.
    qs : List[float]
        The quantiles to compute.
    add_ptp : bool, optional
        Whether to add the peak-to-peak value as the last quantile, by default False.
        Note that the first (i.e., 0 = min) and land (i.e., 1 = max) quantiles should
        be included in `qs` if `add_ptp` is True.
    add_iqr : bool, optional
        Whether to add the interquartile range as the last value, by default False.
        Note that the first (i.e., 0.25 = q1) and land (i.e., 0.75 = q3) quantiles
        should be included in `qs` if `add_iqr` is True.
        If both `add_ptp` and `add_iqr` are True, the interquartile range will be
        appended tot the values as final value (with the peak-to-peak value before it).

    Returns
    -------
    np.ndarray
        of shape: [n_rows, len(qs) + add_ptp]

    """
    q_values = np.quantile(x, qs, axis=axis)
    if add_ptp:
        # the first and last quantile are 0 (i.e., min) and 1 (i.e., max)
        assert qs[0] == 0 and qs[-1] == 1
        q_values = np.concatenate(
            [q_values, (q_values[1] - q_values[0])[None, :]], axis=0
        )
    if add_iqr:
        # the first and last quantile are 0.25 (i.e., q1) and 0.75 (i.e., q3)
        assert 0.25 in qs and 0.75 in qs
        q1 = q_values[qs.index(0.25)]
        q3 = q_values[qs.index(0.75)]
        q_values = np.concatenate([q_values, (q3 - q1)[None, :]], axis=0).astype(
            x.dtype
        )
    return q_values.T

In [ ]:
f_abs_mean_std = FuncWrapper(
    func=abs_mean_std_vect,
    vectorized=True,
    output_names=['abs_mean', 'abs_std']
)

f_qs_vect = FuncWrapper(
    quantiles_vect,
    vectorized=True,
    axis=-1,
    qs=[0, 0.1, 0.25, 0.5, 0.75, 0.9, 1],
    add_ptp=True,
    add_iqr=True,
    output_names=[
        "min",
        "q_0.1",
        "q_0.25",
        "median",
        "q_0.75",
        "q_0.9",
        "max",
        "ptp",
        "iqr",
    ],
) 

last = FuncWrapper(func=lambda x: x[:, -1], 
                   vectorized=True, 
                   output_names=['last'])
first = FuncWrapper(func=lambda x: x[:, 0], 
                    vectorized=True, 
                    output_names=['first'])
diff = FuncWrapper(func=lambda x: np.diff(x, axis=-1)[:, -1].flatten(), 
                   vectorized=True, 
                   output_names=['diff'])
f_std = FuncWrapper(func=std_vect, 
                    vectorized=True, 
                    output_names=['std'])
f_mean = FuncWrapper(func=mean_vect, 
                     vectorized=True, 
                     output_names=['mean'])
# f_min_max = FuncWrapper(func=min_max_vect, vectorized=True, output_names=['min', 'max'])

f_skew_v = FuncWrapper(ss.skew, 
                       vectorized=True, 
                       output_names="skew", axis=-1)
f_kurt_v = FuncWrapper(ss.kurtosis, 
                       vectorized=True, 
                       output_names="kurt", axis=-1)

In [ ]:
s_names = [
    #     # The torque sensors
    #     # "Motors.Roller1.Torque",
    # "Motors.Dancer1.Torque",
    # "Motors.Dancer1.Position",
    #     # "Motors.Dancer1.Speed",
    #     # "Motors.Roller1.WebSpeedSimple",
    #     "Motors.Roller1.SlipSimple",
    # "Motors.Traction1.Torque",
    "Traction1_diff(speed)",
    # "Motors.Traction1.Speed",
    # "Motors.Traction1.Setpoint_diff",
    # "Motors.Traction1.Position_diff",
    # "Motors.Traction1.Speed",  # is good
    # "Motors.Traction1.Setpoint", # setpoint and speed are 99.99% correlated
    # "Motors.Traction2.Torque",
    # "Motors.Traction2.Speed",  # is good
    #     # "Motors.Accumulator.Torque",
    #     # and the load cell sensor
    # "AnalogSensors.LoadCell1",
    # "AnalogSensors.LoadCell2",
    #     # The engineered signals
    #     # "Dancer1_diff(delta(torque, speed))",
    # "Dancer1_delta(torque, speed)",  # TODO -> has speed incorporated
    "Traction1_diff(delta(torque, speed))",  # is good
    # "Traction2_diff(delta(torque, speed))", # is good
    "delta_traction_speed",
    # "Traction1_delta(torque,diff(speed))",
    "LoadCell1_diff",
    "LoadCell2_diff",
    "diff(delta(LoadCell1, LoadCell2))",
    "delta(LoadCell1, LoadCell2)",
]

# Append s_names with all gain factor versions of the current signals in s_names
# s_names += [f"{col}_gain" for col in s_names]
# Add the gain factor itself as a signal to s_names
# s_names += ["gain_factor"]
# Remove all signals that don't contain "gain"
# s_names = [col for col in s_names if "gain" in col]


granular_s_names = [
    # "Motors.Traction1.Setpoint_diff",
    # "Motors.Traction1.Position_diff",
    # "Motors.Traction2.Setpoint_diff",
    # "Motors.Traction2.Position_diff",
]
# granular_s_names += [f"{col}_gain" for col in granular_s_names]
# Add the gain factor itself as a signal to granular_s_names
# granular_s_names += ["gain_factor"]
# Remove all signals that don't contain "gain"
# granular_s_names = [col for col in granular_s_names if "gain" in col]


diff_s_names = [
    "Motors.Dancer1.Position",
    "Motors.Dancer1.Speed",
    # "Motors.Accumulator.Position",
    # "Motors.Accumulator.Speed",
    "Motors.Roller1.Position",
    "Motors.Roller1.Speed",
    "Motors.Traction1.Position",
    "Motors.Traction1.Setpoint",
    "Motors.Traction2.Position",
    "Motors.Traction2.Setpoint",
    # "AnalogSensors.LoadCell1",
]
# diff_s_names += [f"{col}_gain" for col in diff_s_names]
# Remove all signals that don't contain "gain"
# diff_s_names = [col for col in diff_s_names if "gain" in col]

# s_names = list(set(df_train.columns).difference({'File', 'Slip'}))

STRIDE = 1
fc = FeatureCollection(
    feature_descriptors=[
        MultipleFeatureDescriptors(
            functions=[f_std, f_mean],  # , f_skew_v, f_kurt_v],
            series_names=s_names,
            windows=[4, 8, 16, 32, 64],
            strides=STRIDE,
        ),
        # MultipleFeatureDescriptors(
        #     functions=[f_qs_vect], # f_skew_v, f_kurt_v],
        #     series_names=s_names,
        #     windows=[16, 32, 64],
        #     strides=STRIDE,
        # ),
        MultipleFeatureDescriptors(
            functions=[last, first, diff],
            series_names=s_names + granular_s_names,
            windows=[2],
            strides=STRIDE,
        ),
        MultipleFeatureDescriptors(
            functions=[diff], series_names=diff_s_names, windows=[2], strides=STRIDE
        ),
        MultipleFeatureDescriptors(
            functions=[first],
            series_names=["UserInput.WebDirection"],
            windows=[1],
            strides=STRIDE,
        ),
    ]
)

In [12]:
df_feat_train_ = fc.calculate(df_train, show_progress=True, n_jobs=None, return_df=True)
df_feat_train_ = df_feat_train_.reset_index(drop=False)
df_feat_train_ = df_feat_train_.dropna()  # we drop the nan values so that we can use sklearn later on

df_feat_val_ = fc.calculate(df_validation, show_progress=True, n_jobs=None, return_df=True)
df_feat_val_ = df_feat_val_.reset_index(drop=False)
df_feat_val_ = df_feat_val_.dropna()  # we drop the nan values so that we can use sklearn later on

df_feat_test_ = fc.calculate(df_test, show_progress=True, n_jobs=None, return_df=True)
df_feat_test_ = df_feat_test_.reset_index(drop=False)
df_feat_test_ = df_feat_test_.dropna()

100%|██████████| 100/100 [00:01<00:00, 58.49it/s]


In [13]:
def get_shift_config(df, w: int, time_col, shift_type: str = "next"):
    w_cols = seq(df.columns).filter(lambda x: re.search(f"_w={w}$", x)).to_list()
    if shift_type.startswith("prev"):
        number = shift_type.lstrip("prev")
        number = 1 if len(number) == 0 else int(number)
        f_df = df.loc[:, w_cols].add_suffix(f"_prev{number}").copy()
        f_df[time_col] = df[time_col] + number * w
        return f_df
    if shift_type.startswith("next"):
        number = shift_type.lstrip("next")
        number = 1 if len(number) == 0 else int(number)
        f_df = df.loc[:, w_cols].add_suffix(f"_next{number}").copy()
        f_df[time_col] = df[time_col] - number * w
        return f_df
    else:
        raise ValueError(f"Unknown shift: {shift_type}")

In [14]:
df_feat_train = df_feat_train_
df_feat_val = df_feat_val_
df_feat_test = df_feat_test_

for w_size, shift in [
    (2, "next"),
    (2, "next2"),
    (2, "next4"),
    (2, "next8"),
    # (2, "next16"),

    (4, "prev"),
    (4, "prev2"),
    (4, "next"),
    (8, "prev"),
    (8, "prev2"),
    (8, "next2"),

    (2, "prev"),
    (2, "prev2"),
    (2, "prev4"),
    (2, "prev8"),
    (2, "prev12"),
    (2, "prev16"),
    # 
    # (4, "prev"),
    # (8, "prev"),
    # (16, "prev"),
    # (4, "next"),
    # (8, "next"),
    # (16, "next"),
]:  # , (4, "next")]: #, (8, "next"), (16, "next")]:
    df_feat_train = df_feat_train.merge(
        get_shift_config(df_feat_train_, w_size, TIME_COL, shift), on=TIME_COL
    )
    df_feat_val = df_feat_val.merge(
        get_shift_config(df_feat_val_, w_size, TIME_COL, shift), on=TIME_COL
    )
    df_feat_test = df_feat_test.merge(
        get_shift_config(df_feat_test, w_size, TIME_COL, shift), on=TIME_COL
    )

In [15]:
# Create the final feature dataframes by merging with the original dataframes
df_feat_train_m = pd.merge(
    df_feat_train.set_index(TIME_COL),
    df_train[[GROUP_COL, TARGET_COL] + ['Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']],
    left_index=True,
    right_index=True,
    how="left",
)
df_feat_val_m = pd.merge(
    df_feat_val.set_index(TIME_COL),
    df_validation[[GROUP_COL, TARGET_COL] + ['Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']],
    left_index=True,
    right_index=True,
    how="left",
)
df_feat_test_m = pd.merge(
    df_feat_test.set_index(TIME_COL),
    df_test[[GROUP_COL, TARGET_COL] + ['Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']],
    left_index=True,
    right_index=True,
    how="left",
)

In [16]:
feat_cols = list(set(df_feat_train_m.columns) - {TIME_COL, GROUP_COL, TARGET_COL} - set(['Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']))
print(f"Number of features: {len(feat_cols)}")

Number of features: 474


In [17]:
sel_catboost_df = pd.read_csv('./../data/features/selected_features_catboost.csv')
sel_linear_df = pd.read_csv('./../data/features/selected_features_linear.csv')
selected_features = list(set(sel_catboost_df['selected_features'].to_list()).union(set(sel_linear_df['selected_features'].to_list())))
print(f"Number of selected features: {len(selected_features)}")

Number of selected features: 14


In [18]:
def evaluate_model(model, cols = feat_cols, pulse_speed_th = 50):
    high_speed_train = df_feat_train_m.loc[df_feat_train_m['PulseSpeed'] >= pulse_speed_th]
    low_speed_train = df_feat_train_m.loc[df_feat_train_m['PulseSpeed'] < pulse_speed_th]
    high_speed_val = df_feat_val_m.loc[df_feat_val_m['PulseSpeed'] >= pulse_speed_th]
    low_speed_val = df_feat_val_m.loc[df_feat_val_m['PulseSpeed'] < pulse_speed_th]
    high_speed_50_test = df_feat_test_m.loc[df_feat_test_m['PulseSpeed'] == pulse_speed_th]
    high_speed_60_test = df_feat_test_m.loc[df_feat_test_m['PulseSpeed'] == 60]
    low_speed_test = df_feat_test_m.loc[df_feat_test_m['PulseSpeed'] < pulse_speed_th]

    print(f"Low speed Train Shape: {low_speed_train[cols].shape}, High speed Train Shape: {high_speed_train[cols].shape}")
    print(f"Low speed Val Shape: {low_speed_val[cols].shape}, High speed Val Shape: {high_speed_val[cols].shape}")
    print(f"Low speed Test Shape: {low_speed_test[cols].shape}, High speed (50) Test Shape: {high_speed_50_test[cols].shape}, High speed (60) Test Shape: {high_speed_60_test[cols].shape}")

    print(f"Low speed Train   MSE: {mean_squared_error(low_speed_train[TARGET_COL], model.predict(low_speed_train[cols])):.4f}", end=" | ")
    print(f"High speed Train  MSE: {mean_squared_error(high_speed_train[TARGET_COL], model.predict(high_speed_train[cols])):.4f}", end="\n")
    print(f"Low speed Val     MSE: {mean_squared_error(low_speed_val[TARGET_COL], model.predict(low_speed_val[cols])):.4f}", end=" | ",)
    print(f"High speed Val    MSE: {mean_squared_error(high_speed_val[TARGET_COL], model.predict(high_speed_val[cols])):.4f}", end="\n")
    print(f"Low speed Test    MSE: {mean_squared_error(low_speed_test[TARGET_COL], model.predict(low_speed_test[cols])):.4f}", end=" | ")
    print(f"High speed (50) Test   MSE: {mean_squared_error(high_speed_50_test[TARGET_COL], model.predict(high_speed_50_test[cols])):.4f}", end=" | ")
    print(f"High speed (60) Test   MSE: {mean_squared_error(high_speed_60_test[TARGET_COL], model.predict(high_speed_60_test[cols])):.4f}", end="\n")


def evaluate_model_quantile(model, cols = feat_cols, avg_traction_accl_th = df_train['avg_traction1_accel'].quantile(0.975)):
    assert STRIDE == 1

    high_speed_mask = df_train[(df_train['avg_traction1_accel'] >= avg_traction_accl_th)].index
    low_speed_mask = df_train[(df_train['avg_traction1_accel'] < avg_traction_accl_th)].index
    high_speed_train = df_feat_train_m.loc[high_speed_mask.intersection(df_feat_train_m.index)]
    low_speed_train = df_feat_train_m.loc[low_speed_mask.intersection(df_feat_train_m.index)]

    high_speed_mask = df_validation[(df_validation['avg_traction1_accel'] >= avg_traction_accl_th)].index
    low_speed_mask = df_validation[(df_validation['avg_traction1_accel'] < avg_traction_accl_th)].index
    high_speed_val = df_feat_val_m.loc[high_speed_mask.intersection(df_feat_val_m.index)]
    low_speed_val = df_feat_val_m.loc[low_speed_mask.intersection(df_feat_val_m.index)]

    high_speed_mask = df_test[(df_test['avg_traction1_accel'] >= avg_traction_accl_th)].index
    low_speed_mask = df_test[(df_test['avg_traction1_accel'] < avg_traction_accl_th)].index
    high_speed_test = df_feat_test_m.loc[high_speed_mask.intersection(df_feat_test_m.index)]
    low_speed_test = df_feat_test_m.loc[low_speed_mask.intersection(df_feat_test_m.index)]

    print(f"Low speed Train Shape: {low_speed_train[cols].shape}, High speed Train Shape: {high_speed_train[cols].shape}")
    print(f"Low speed Val Shape: {low_speed_val[cols].shape}, High speed Val Shape: {high_speed_val[cols].shape}")
    print(f"Low speed Test Shape: {low_speed_test[cols].shape}, High speed Test Shape: {high_speed_test[cols].shape}")

    print(f"Low speed Train   MSE: {mean_squared_error(low_speed_train[TARGET_COL], model.predict(low_speed_train[cols])):.4f}", end=" | ")
    print(f"High speed Train  MSE: {mean_squared_error(high_speed_train[TARGET_COL], model.predict(high_speed_train[cols])):.4f}", end="\n")
    print(f"Low speed Val     MSE: {mean_squared_error(low_speed_val[TARGET_COL], model.predict(low_speed_val[cols])):.4f}", end=" | ",)
    print(f"High speed Val    MSE: {mean_squared_error(high_speed_val[TARGET_COL], model.predict(high_speed_val[cols])):.4f}", end="\n")
    print(f"Low speed Test    MSE: {mean_squared_error(low_speed_test[TARGET_COL], model.predict(low_speed_test[cols])):.4f}", end=" | ")
    print(f"High speed Test   MSE: {mean_squared_error(high_speed_test[TARGET_COL], model.predict(high_speed_test[cols])):.4f}", end="\n")

In [19]:
# Train on all speeds with selected features as benchmark
X_train, y_train = df_feat_train_m[selected_features], df_feat_train_m[TARGET_COL]
X_val, y_val = df_feat_val_m[selected_features], df_feat_val_m[TARGET_COL]

In [21]:
# Train CatBoost Regressor
catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.07,
    depth=6,
    loss_function='RMSE',
    use_best_model=False,      # Disable best model selection
    verbose=100,
    task_type="CPU",
    random_seed=42,
    bootstrap_type="No",       # Deterministic training
    thread_count=1,             # Disable multithreading randomness
    random_strength=0,         # Disable randomness in feature selection
)

catboost_model.fit(X_train, y_train, eval_set=(X_val, y_val), plot=True)

print(f"MSE: {mean_squared_error(y_val, catboost_model.predict(X_val))}")
print('speed based evaluation')
evaluate_model(catboost_model, cols=selected_features)
print('--'*40)
print('quantile evaluation')
evaluate_model_quantile(catboost_model, cols=selected_features, avg_traction_accl_th=df_train['avg_traction1_accel'].quantile(0.975))

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.8652996	test: 0.8501270	best: 0.8501270 (0)	total: 180ms	remaining: 1m 29s
100:	learn: 0.2235730	test: 0.3047190	best: 0.3047190 (100)	total: 1.88s	remaining: 7.41s
200:	learn: 0.1807488	test: 0.2988663	best: 0.2987717 (197)	total: 3.47s	remaining: 5.17s
300:	learn: 0.1598807	test: 0.2951676	best: 0.2950577 (298)	total: 5.94s	remaining: 3.93s
400:	learn: 0.1481833	test: 0.2943887	best: 0.2942552 (336)	total: 8.88s	remaining: 2.19s
499:	learn: 0.1407218	test: 0.2942359	best: 0.2942359 (499)	total: 10.7s	remaining: 0us

bestTest = 0.2942358788
bestIteration = 499

MSE: 0.08657475169286336
speed based evaluation
Low speed Train Shape: (40660, 14), High speed Train Shape: (13578, 14)
Low speed Val Shape: (40764, 14), High speed Val Shape: (13620, 14)
Low speed Test Shape: (40621, 14), High speed (50) Test Shape: (13636, 14), High speed (60) Test Shape: (13578, 14)
Low speed Train   MSE: 0.0169 | High speed Train  MSE: 0.0284
Low speed Val     MSE: 0.0550 | High speed Val    MSE

In [22]:
mse_list = []
rounds = 2
for i in range(rounds):
    linear_model = Pipeline([
        # ('scaling', QuantileTransformer(output_distribution='normal')),
        ('scaling', StandardScaler()),
        # ("scaling", PowerTransformer()),
        # ('regressor', SGDRegressor(max_iter=10_000))
        # ('regressor', Line(max_iter=10_000))
        # ("regressor", LinearRegression()),
        # ("regressor", Ridge()),
        ("regressor", Lasso(random_state=42)),
        # ("regressor", SVR(max_iter=100))
        # ("regressor", LinearSVR(max_iter))
    ])

    linear_model.fit(X_train, y=y_train, regressor__sample_weight= y_train.abs())

    y_val_pred = linear_model.predict(X_val)
    mse_list.append(mean_squared_error(y_val, y_val_pred))
print(f'mean mse over {rounds} rounds: {np.mean(mse_list)} +/- {np.std(mse_list)}')
print("speed based evaluation")
evaluate_model(linear_model, cols=selected_features)
print("--" * 40)
print("quantile evaluation")
evaluate_model_quantile(
    linear_model,
    cols=selected_features,
    avg_traction_accl_th=df_train["avg_traction1_accel"].quantile(0.975),
)

mean mse over 2 rounds: 0.14014808008379973 +/- 0.0
speed based evaluation
Low speed Train Shape: (40660, 14), High speed Train Shape: (13578, 14)
Low speed Val Shape: (40764, 14), High speed Val Shape: (13620, 14)
Low speed Test Shape: (40621, 14), High speed (50) Test Shape: (13636, 14), High speed (60) Test Shape: (13578, 14)
Low speed Train   MSE: 0.1076 | High speed Train  MSE: 0.2241
Low speed Val     MSE: 0.0949 | High speed Val    MSE: 0.2757
Low speed Test    MSE: 0.0988 | High speed (50) Test   MSE: 0.2804 | High speed (60) Test   MSE: 1.8740
--------------------------------------------------------------------------------
quantile evaluation
Low speed Train Shape: (52879, 14), High speed Train Shape: (1359, 14)
Low speed Val Shape: (53007, 14), High speed Val Shape: (1377, 14)
Low speed Test Shape: (65774, 14), High speed Test Shape: (2061, 14)
Low speed Train   MSE: 0.1212 | High speed Train  MSE: 0.7431
Low speed Val     MSE: 0.0974 | High speed Val    MSE: 1.7846
Low speed

In [23]:
avg_traction_accl_th = df_train['avg_traction1_accel'].quantile(0.975)
high_speed_mask = df_train[(df_train['avg_traction1_accel'] >= avg_traction_accl_th)].index
low_speed_mask = df_train[(df_train['avg_traction1_accel'] < avg_traction_accl_th)].index
high_speed_train = df_feat_train_m.loc[high_speed_mask.intersection(df_feat_train_m.index)]
low_speed_train = df_feat_train_m.loc[low_speed_mask.intersection(df_feat_train_m.index)]

high_speed_mask = df_validation[(df_validation['avg_traction1_accel'] >= avg_traction_accl_th)].index
low_speed_mask = df_validation[(df_validation['avg_traction1_accel'] < avg_traction_accl_th)].index
high_speed_val = df_feat_val_m.loc[high_speed_mask.intersection(df_feat_val_m.index)]
low_speed_val = df_feat_val_m.loc[low_speed_mask.intersection(df_feat_val_m.index)]

high_speed_mask = df_test[(df_test['avg_traction1_accel'] >= avg_traction_accl_th)].index
low_speed_mask = df_test[(df_test['avg_traction1_accel'] < avg_traction_accl_th)].index
high_speed_test = df_feat_test_m.loc[high_speed_mask.intersection(df_feat_test_m.index)]
low_speed_test = df_feat_test_m.loc[low_speed_mask.intersection(df_feat_test_m.index)]

print(f"Low speed Train Shape: {low_speed_train[selected_features].shape}, High speed Train Shape: {high_speed_train[selected_features].shape}")
print(f"Low speed Val Shape: {low_speed_val[selected_features].shape}, High speed Val Shape: {high_speed_val[selected_features].shape}")
print(f"Low speed Test Shape: {low_speed_test[selected_features].shape}, High speed Test Shape: {high_speed_test[selected_features].shape}")

pred_dict = {}
train_catboost_preds = pd.Series(catboost_model.predict(low_speed_train[selected_features]), index=low_speed_train.index)
train_linear_preds = pd.Series(linear_model.predict(high_speed_train[selected_features]), index=high_speed_train.index)
pred_dict['Train'] = pd.concat([train_catboost_preds, train_linear_preds]).sort_index()
valid_catboost_preds = pd.Series(catboost_model.predict(low_speed_val[selected_features]), index=low_speed_val.index)
valid_linear_preds = pd.Series(linear_model.predict(high_speed_val[selected_features]), index=high_speed_val.index)
pred_dict['Validation'] = pd.concat([valid_catboost_preds, valid_linear_preds]).sort_index()
test_catboost_preds = pd.Series(catboost_model.predict(low_speed_test[selected_features]), index=low_speed_test.index)
test_linear_preds = pd.Series(linear_model.predict(high_speed_test[selected_features]), index=high_speed_test.index)
pred_dict['Test'] = pd.concat([test_catboost_preds, test_linear_preds]).sort_index()

print("Quantile evaluation trained using low speed for CatBoost and high speed for Linear")
print(f"Low speed Train   MSE: {mean_squared_error(low_speed_train[TARGET_COL], catboost_model.predict(low_speed_train[selected_features])):.4f}", end=" | ")
print(f"High speed Train  MSE: {mean_squared_error(high_speed_train[TARGET_COL], linear_model.predict(high_speed_train[selected_features])):.4f}", end="\n")
print(f"Low speed Val     MSE: {mean_squared_error(low_speed_val[TARGET_COL], catboost_model.predict(low_speed_val[selected_features])):.4f}", end=" | ",)
print(f"High speed Val    MSE: {mean_squared_error(high_speed_val[TARGET_COL], linear_model.predict(high_speed_val[selected_features])):.4f}", end="\n")
print(f"Low speed Test    MSE: {mean_squared_error(low_speed_test[TARGET_COL], catboost_model.predict(low_speed_test[selected_features])):.4f}", end=" | ")
print(f"High speed Test   MSE: {mean_squared_error(high_speed_test[TARGET_COL], linear_model.predict(high_speed_test[selected_features])):.4f}", end="\n")

Low speed Train Shape: (52879, 14), High speed Train Shape: (1359, 14)
Low speed Val Shape: (53007, 14), High speed Val Shape: (1377, 14)
Low speed Test Shape: (65774, 14), High speed Test Shape: (2061, 14)
Quantile evaluation trained using low speed for CatBoost and high speed for Linear
Low speed Train   MSE: 0.0184 | High speed Train  MSE: 0.7431
Low speed Val     MSE: 0.0479 | High speed Val    MSE: 1.7846
Low speed Test    MSE: 0.2894 | High speed Test   MSE: 8.5814


In [24]:
# Create a new dictionary and call it pred_feat_dict with features
pred_feat_dict = {
    'Train': df_feat_train_m[selected_features + [TARGET_COL, 'Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']],
    'Validation': df_feat_val_m[selected_features + [TARGET_COL, 'Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']],
    'Test': df_feat_test_m[selected_features + [TARGET_COL, 'Set', 'PulseSpeed', 'PulseRampTime', 'PulseDirection', 'PulseNumber']],
}
# Add prediction to the new dictionary
for key in pred_dict.keys():
    pred_feat_dict[key]['Prediction'] = pred_dict[key]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13280\3197929552.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13280\3197929552.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13280\3197929552.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https:/

In [25]:
high_speed = 50
print(f"Low speed Train Shape: {pred_feat_dict['Train'].loc[pred_feat_dict['Train']['PulseSpeed'] < high_speed][selected_features].shape}, High speed Train Shape: {pred_feat_dict['Train'].loc[pred_feat_dict['Train']['PulseSpeed'] >= high_speed][selected_features].shape}")
print(f"Low speed Val Shape: {pred_feat_dict['Validation'].loc[pred_feat_dict['Validation']['PulseSpeed'] < high_speed][selected_features].shape}, High speed Val Shape: {pred_feat_dict['Validation'].loc[pred_feat_dict['Validation']['PulseSpeed'] >= high_speed][selected_features].shape}")
print(f"Low speed Test Shape: {pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] < high_speed][selected_features].shape}, High speed (50) Test Shape: {pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] == high_speed][selected_features].shape}, High speed (60) Test Shape: {pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] == 60][selected_features].shape}")

print(f"Low speed Train   MSE: {mean_squared_error(pred_feat_dict['Train'].loc[pred_feat_dict['Train']['PulseSpeed'] < high_speed][TARGET_COL], pred_feat_dict['Train'].loc[pred_feat_dict['Train']['PulseSpeed'] < high_speed]['Prediction']):.4f}", end=" | ")
print(f"High speed Train  MSE: {mean_squared_error(pred_feat_dict['Train'].loc[pred_feat_dict['Train']['PulseSpeed'] >= high_speed][TARGET_COL], pred_feat_dict['Train'].loc[pred_feat_dict['Train']['PulseSpeed'] >= high_speed]['Prediction']):.4f}", end="\n")
print(f"Low speed Val     MSE: {mean_squared_error(pred_feat_dict['Validation'].loc[pred_feat_dict['Validation']['PulseSpeed'] < high_speed][TARGET_COL], pred_feat_dict['Validation'].loc[pred_feat_dict['Validation']['PulseSpeed'] < high_speed]['Prediction']):.4f}", end=" | ",)
print(f"High speed Val    MSE: {mean_squared_error(pred_feat_dict['Validation'].loc[pred_feat_dict['Validation']['PulseSpeed'] >= high_speed][TARGET_COL], pred_feat_dict['Validation'].loc[pred_feat_dict['Validation']['PulseSpeed'] >= high_speed]['Prediction']):.4f}", end="\n")
print(f"Low speed Test    MSE: {mean_squared_error(pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] < high_speed][TARGET_COL], pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] < high_speed]['Prediction']):.4f}", end=" | ")
print(f"High speed (50) Test   MSE: {mean_squared_error(pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] == high_speed][TARGET_COL], pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] == high_speed]['Prediction']):.4f}", end=" | ")
print(f"High speed (60) Test   MSE: {mean_squared_error(pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] == 60][TARGET_COL], pred_feat_dict['Test'].loc[pred_feat_dict['Test']['PulseSpeed'] == 60]['Prediction']):.4f}", end="\n")


Low speed Train Shape: (40660, 14), High speed Train Shape: (13578, 14)
Low speed Val Shape: (40764, 14), High speed Val Shape: (13620, 14)
Low speed Test Shape: (40621, 14), High speed (50) Test Shape: (13636, 14), High speed (60) Test Shape: (13578, 14)
Low speed Train   MSE: 0.0220 | High speed Train  MSE: 0.0802
Low speed Val     MSE: 0.0604 | High speed Val    MSE: 0.1859
Low speed Test    MSE: 0.0643 | High speed (50) Test   MSE: 0.2009 | High speed (60) Test   MSE: 2.3104


In [ ]:
# Add predictions to each set in set_dict
set_dict = {name: g.assign(pred=pred_dict[name]) for name, g in set_dict.items()}
gap_dict = {ramp_time[0]:g.shape[0] for ramp_time, g in set_dict['Test'].groupby(['PulseRampTime'])}

In [27]:
def pad_dataframe_zeros(df: pd.DataFrame, num_zeros: dict) -> pd.DataFrame:
    """
    """
    # First add rows equal to the gap
    zero_rows = pd.DataFrame(0, index=range(num_zeros), columns=df.columns)
    # Concatenate with the original DataFrame
    padded_df = pd.concat([df, zero_rows], ignore_index=True)
    return padded_df

In [ ]:
row_order_dict = {'Train': 1, 'Validation': 2, 'Test': 3}
col_order_dict = {8:1, 6:2, 4:3, 2:4}
subplot_list = [f'{r[0]}_{r[1]}s' for r in itertools.product(row_order_dict.keys(), col_order_dict.keys())]
fig = FigureResampler(make_subplots(rows=3, cols=4, subplot_titles=subplot_list, shared_xaxes='all', shared_yaxes='all'))
for unique_set, set_g in set_dict.items():
    for ramp_time, g in set_g.groupby('PulseRampTime'):
        if unique_set != 'Test':
            g = pad_dataframe_zeros(g, gap_dict[ramp_time]-g.shape[0])
        else:
            g = g.reset_index()

        # fig.add_trace(go.Scatter(name=unique_set + '_WebSpeedReal_' + str(ramp_time) + '_s', line_color='blue'), hf_x=g.index, hf_y=g['WebSpeedReal'], row=row_order_dict[unique_set], col=col_order_dict[ramp_time])
        fig.add_trace(go.Scatter(name=unique_set + '_Slip_' + str(ramp_time) + '_s', line_color='red'),hf_x=g.index, hf_y=g['Slip'], row=row_order_dict[unique_set], col=col_order_dict[ramp_time])
        # Add predictions
        fig.add_trace(go.Scatter(name=unique_set + '_pred_' + str(ramp_time) + '_s', line_color='green'), hf_x=g.index, hf_y=g['pred'], row=row_order_dict[unique_set], col=col_order_dict[ramp_time])
        # Add vertical lines between each speed
        for (speed), cur_df in g.groupby('PulseSpeed'):
            fig.add_vline(x=cur_df.index[-1], line_width=1, line_dash='dash', line_color='black', row=row_order_dict[unique_set], col=col_order_dict[ramp_time])

for row in range(1, 4):
    for col in range(1, 5):
        fig.update_yaxes(tickvals=[-60, -50, -40, -30, -20, 0, 20, 30, 40, 50, 60], row=row, col=col)

for col in range(1, 5):
    fig.update_xaxes(title='Time', row=3, col=col)

for row in range(1, 4):
    fig.update_yaxes(title='Slip (m/min)', row=row, col=1)

fig.update_layout(height=800, template='plotly_white')
# Remove legend
fig.update_layout(showlegend=False)
fig.show()
# Save figure as pdf
fig.write_image(format="pdf", file="./../figures/ramp_pulses_preds.pdf", height=800, width=1100)

In [30]:
# Plot a figure of the predictions of each subset with a ramp time of 2s
for subset in set_dict.keys():
    fig = FigureResampler(make_subplots(rows=1, cols=1, subplot_titles=[subset + ' set predictions for ramp time 2s'], shared_xaxes='all', shared_yaxes='all'))
    ramp_time = 2
    set_g = set_dict[subset]
    g = set_g[set_g['PulseRampTime'] == ramp_time]
    g = g.reset_index()
    # Add webspeed real as second y axis
    fig.add_trace(go.Scatter(name=subset + '_WebSpeedReal_' + str(ramp_time) + '_s', line_color='blue', yaxis='y2'), hf_x=g.index, hf_y=g['WebSpeedReal'], row=1, col=1)

    fig.add_trace(go.Scatter(name=subset + '_Slip_' + str(ramp_time) + '_s', line_color='red'),hf_x=g.index, hf_y=g['Slip'], row=1, col=1)
    fig.add_trace(go.Scatter(name=subset + '_pred_' + str(ramp_time) + '_s', line_color='green'), hf_x=g.index, hf_y=g['pred'], row=1, col=1)
    # Add vertical lines between each speed
    for (speed), cur_df in g.groupby('PulseSpeed'):
        fig.add_vline(x=cur_df.index[-1], line_width=1, line_dash='dash', line_color='black', row=1, col=1)
    fig.update_xaxes(title='Time', row=1, col=1)
    fig.update_yaxes(title='Slip (m/min)', row=1, col=1)
    fig.update_yaxes(range=[-65, 65], tickvals=[-60, -50, -40, -30, -20, 0, 20, 30, 40, 50, 60], row=1, col=1)
    fig.update_layout(height=400, template='plotly_white')
    # Remove legend
    fig.update_layout(showlegend=False)
    fig.show()
    # Save figure as pdf
    fig.write_image(format="pdf", file="./../figures/" + subset.lower() + "_2s_preds.pdf", height=400, width=1100)